# Capstone Project: Data Pipeline and Preprocessing
**Day 48 of 60 Days of Data Science**

This notebook demonstrates the preprocessing pipeline for our Customer Intelligence Platform.

## 1. Introduction and Setup
We'll load the raw data, perform data quality checks, and set up our custom transformers for a scikit-learn pipeline.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import sys

# Add src to path to import our custom transformers
sys.path.append('src')
from preprocessing import OutlierClipper, DateFeaturesExtractor, FeatureAggregator

# Setup visualization
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

## 2. Data Loading and Exploration
Load the raw customer and transactional data.

In [ ]:
df_customers = pd.read_csv('data/raw/customers.csv')
df_transactions = pd.read_csv('data/raw/transactions.csv')

print("Customers shape:", df_customers.shape)
print("Transactions shape:", df_transactions.shape)
df_customers.head()

## 3. Preprocessing Transaction Data
We will extract date features and then aggregate transaction amounts per customer.

In [ ]:
# Transaction Pipeline
transaction_pipeline = Pipeline([
    ('date_features', DateFeaturesExtractor(date_col='transaction_date'))
])

df_transactions_processed = transaction_pipeline.fit_transform(df_transactions)

# Aggregate transactions to customer level
aggregator = FeatureAggregator(
    groupby_col='customer_id',
    agg_dict={
        'amount': ['sum', 'mean', 'count'],
        'transaction_date_year': 'max'
    }
)
df_txn_agg = aggregator.fit_transform(df_transactions_processed)
# Flatten MultiIndex columns
df_txn_agg.columns = ['customer_id', 'total_spend', 'avg_transaction_value', 'transaction_count', 'last_transaction_year']
df_txn_agg.head()

## 4. Customer Data Preprocessing Pipeline
We'll build a ColumnTransformer to handle missing values, encode categorical variables, and scale numerical variables.

In [ ]:
# Merge aggregated transactions with customer data
df_merged = df_customers.merge(df_txn_agg, on='customer_id', how='left')

# Define column groups
numeric_features = ['age', 'tenure_months', 'estimated_income', 'total_spend', 'avg_transaction_value', 'transaction_count']
categorical_features = ['gender', 'location']

# Numeric Pipeline: Impute missing values with median, clip outliers, and scale
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('outlier_clipper', OutlierClipper(factor=1.5)),
    ('scaler', StandardScaler())
])

# Categorical Pipeline: Impute with mode and one-hot encode
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Combine into ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# Fit and transform
X_processed = preprocessor.fit_transform(df_merged)

# Get feature names for categorical columns
cat_feature_names = preprocessor.named_transformers_['cat']['onehot'].get_feature_names_out(categorical_features)
all_feature_names = numeric_features + list(cat_feature_names)

# Convert back to DataFrame
df_cleaned = pd.DataFrame(X_processed, columns=all_feature_names)
df_cleaned['customer_id'] = df_merged['customer_id'].values

df_cleaned.head()

## 5. Saving Cleaned Data
Save the finalized data to the `data/cleaned/` directory.

In [ ]:
import os
os.makedirs('data/cleaned', exist_ok=True)
df_cleaned.to_csv('data/cleaned/customers_preprocessed.csv', index=False)
print("Saved preprocessed dataset.")